# Experiments with a single-column atmosphere

In this notebook you will inspect the model grid, run a short atmospheric column experiment, perturb carbon dioxide, compare convection schemes, and test parameter sensitivity. All teaching settings are defined below; no external configuration file is required.

These short runs demonstrate mechanisms and workflow. They are not equilibrated climates and must not be interpreted as estimates of equilibrium climate sensitivity. The long research configurations remain in `scm/configs/`.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments.ipynb)

## Colab setup

Run this cell first. It downloads and installs the current course model into the Colab runtime. A fresh Colab runtime needs to run this cell once.

In [ ]:
from pathlib import Path
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11 or newer is required.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[plot]',
], check=True)
print('SCM ready from', root)

In [ ]:
from copy import deepcopy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from scm.column_model import initial_state, run
from scm.diagnostics import equilibrium_stats
from scm.ensemble import default_params
from scm.thermo import make_grid

torch.manual_seed(0)
device = torch.device('cpu')
print('device:', device)

## The complete teaching configuration

The dictionary below is the control panel for every experiment in this notebook. Keeping it here makes each choice visible and easy to change.

In [ ]:
experiment = {
    'nlevels': 20,
    'dt': 900.0,
    'days': 20,
    'diagnostic_hours': 6,
    'radiation_steps': 8,
    'surface_temperature': 290.0,
    'surface_pressure': 100000.0,
    'co2': 400.0,
    'solar_constant': 1360.0,
    'zenith_factor': 0.25,
    'ocean_depth': 50.0,
    'surface_albedo': 0.28,
    'wind_speed': 5.0,
    'convection': 'mass_flux',
    'radiation': 'semi_gray',
    'slab_ocean': True,
}
experiment

In [ ]:
def makeparams(settings):
    params = default_params(device=device)
    params.update({
        'dt': settings['dt'],
        'ps0': settings['surface_pressure'],
        'ts_init': settings['surface_temperature'],
        'co2': settings['co2'],
        'co2_ref': 400.0,
        'solar_constant': settings['solar_constant'],
        'zenith_factor': settings['zenith_factor'],
        'ocean_depth': settings['ocean_depth'],
        'albedo': settings['surface_albedo'],
        'wind_speed': settings['wind_speed'],
        'convection_scheme': settings['convection'],
        'radiation_scheme': settings['radiation'],
        'use_slab_ocean': settings['slab_ocean'],
    })
    return params

def integrate(settings, state=None):
    grid = make_grid(settings['nlevels'], device=device)
    params = makeparams(settings)
    if state is None:
        state = initial_state(1, grid, params, device=device)
    stepsperday = round(86400 / settings['dt'])
    nsteps = round(settings['days'] * stepsperday)
    diagnosticsteps = max(1, round(settings['diagnostic_hours'] * 3600 / settings['dt']))
    start = time.perf_counter()
    state, history = run(
        state, grid, params, nsteps,
        rad_interval=settings['radiation_steps'],
        diag_interval=diagnosticsteps,
    )
    elapsed = time.perf_counter() - start
    return grid, params, state, history, elapsed

def series(history, name, scale=1.0):
    return np.array([entry[name][0].detach().cpu().item() for entry in history]) * scale

def days(history, dt):
    return np.array([entry['step'] for entry in history]) * dt / 86400

## Exercise 1: inspect the vertical grid

Plot the full-level pressure coordinate. Which part of the atmosphere receives the finest resolution? Why might that be helpful for surface exchange and convection?

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
sigma = grid['sigma_full'].cpu().numpy()
levels = np.arange(experiment['nlevels'])

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(sigma, levels, marker='o')
ax.invert_yaxis()
ax.set(xlabel='pressure divided by surface pressure', ylabel='model level')
ax.grid(alpha=0.3)
plt.show()

## Exercise 2: run the control column

Run the control, then examine surface temperature, top-of-atmosphere energy balance, and precipitation. Explain why a 20-day run should not be called an equilibrium climate.

In [ ]:
grid, controlparams, controlstate, controlhistory, elapsed = integrate(experiment)
controlstats = equilibrium_stats(controlhistory, last_n=min(20, len(controlhistory)))

print(f'elapsed: {elapsed:.1f} s')
print(f"surface temperature: {controlstats['ts_mean'][0].item():.2f} K")
print(f"toa net flux: {controlstats['toa_net_mean'][0].item():+.2f} W m-2")
print(f"precipitation: {controlstats['precip_total_mean'][0].item() * 86400:.2f} mm day-1")

In [ ]:
timeaxis = days(controlhistory, experiment['dt'])
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].plot(timeaxis, series(controlhistory, 'ts'))
axes[0].set_ylabel('surface temperature (K)')
axes[1].plot(timeaxis, series(controlhistory, 'toa_net'))
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('toa net (W m-2)')
axes[2].plot(timeaxis, series(controlhistory, 'precip_total', 86400))
axes[2].set_ylabel('precipitation (mm day-1)')
axes[2].set_xlabel('model day')
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Exercise 3: double carbon dioxide

Branch from the final control state, double CO$_2$, and compare the transient response. Before running the cell, predict the sign of the initial top-of-atmosphere response and the surface-temperature response.

In [ ]:
warmsettings = dict(experiment)
warmsettings['co2'] = 800.0
warmstate = deepcopy(controlstate)
warmgrid, warmparams, warmstate, warmhistory, elapsed = integrate(warmsettings, warmstate)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
warmaxis = days(warmhistory, warmsettings['dt'])
axes[0].plot(warmaxis, series(warmhistory, 'ts'))
axes[0].set_ylabel('surface temperature (K)')
axes[1].plot(warmaxis, series(warmhistory, 'toa_net'))
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(xlabel='days after doubling', ylabel='toa net (W m-2)')
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"temperature change after {warmsettings['days']} days: {warmstate['ts'][0].item() - controlstate['ts'][0].item():+.3f} K")

## Exercise 4: structural uncertainty in convection

Run otherwise identical columns with mass-flux and Betts--Miller convection. Compare precipitation and the final temperature profiles. Which differences are transient, and which suggest a structural response to the parameterization?

In [ ]:
convectionresults = {}
for convection in ['mass_flux', 'betts_miller']:
    settings = dict(experiment)
    settings['convection'] = convection
    convectionresults[convection] = integrate(settings)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for convection, result in convectionresults.items():
    grid, params, state, history, elapsed = result
    axes[0].plot(days(history, experiment['dt']), series(history, 'precip_total', 86400), label=convection)
    axes[1].plot(state['t'][0].cpu(), grid['sigma_full'].cpu(), label=convection)
axes[0].set(xlabel='model day', ylabel='precipitation (mm day-1)')
axes[1].set(xlabel='temperature (K)', ylabel='pressure divided by surface pressure')
axes[1].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

## Exercise 5: parameter sensitivity

Choose one physical parameter, define a small range, and explain your hypothesis before running the experiment. The example varies mixed-layer depth. Replace it with surface albedo, wind speed, or a convection parameter for an extension.

In [ ]:
depths = [20.0, 50.0, 100.0]
depthresults = {}
for depth in depths:
    settings = dict(experiment)
    settings['ocean_depth'] = depth
    grid, params, state, history, elapsed = integrate(settings)
    depthresults[depth] = history

fig, ax = plt.subplots(figsize=(8, 4))
for depth, history in depthresults.items():
    ax.plot(days(history, experiment['dt']), series(history, 'ts'), label=f'{depth:.0f} m')
ax.set(xlabel='model day', ylabel='surface temperature (K)')
ax.grid(alpha=0.3)
ax.legend(title='mixed-layer depth')
plt.show()

## Exercise 6: numerical sensitivity and scientific reporting

Repeat one experiment with 10, 20, and 40 vertical levels or with timesteps of 450 and 900 seconds. Compare a physically relevant diagnostic and runtime. Decide whether the differences are small enough for your scientific question.

For your report, record: the full settings dictionary, your hypothesis, the diagnostic used, whether the column was equilibrated, a figure with units, and a short explanation of the mechanism. Never report the transient warming above as equilibrium climate sensitivity.

In [ ]:
# Your numerical-sensitivity experiment goes here.
# Start by copying experiment, changing one setting, and calling integrate.